# Day 43 — Practice Questions

## K-Means clustering

**Boston University · Data Science Program**

---

**12 questions in four parts.**

| Part | | Questions |
|---|---|---|
| **A** | Running the loop | 1 – 3 |
| **B** | Choosing k | 4 – 6 |
| **C** | Breaking it | 7 – 10 |
| **D** | Using it | 11 – 12 |

Run the setup cell first. Every question is self-contained after that.

> Replace every `...` with your own code. Where a question asks you to *explain*, write
> two or three sentences in the markdown cell — the explanation is the point.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.simplefilter("ignore")

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.datasets import load_iris, load_wine, make_blobs, make_moons

TEAL, CORAL, VIOLET, INK = "#1F9E86", "#E05C45", "#8268B5", "#131F33"
COLS = [TEAL, CORAL, VIOLET, "#3E7CB1", "#F0A530", "#7B8AA1"]

def show(X, labels=None, centres=None, title="", ax=None):
    ax = ax or plt.gca()
    c = "#9AA8BE" if labels is None else [COLS[l % 6] for l in labels]
    ax.scatter(X[:, 0], X[:, 1], c=c, s=20, edgecolors="white", linewidths=.4, zorder=3)
    if centres is not None:
        ax.scatter(centres[:, 0], centres[:, 1], marker="X", s=280, c=INK,
                   edgecolors="white", linewidths=1.6, zorder=6)
    ax.set_title(title, fontsize=11)
    ax.set_xticks([]); ax.set_yticks([])

# ---- Part A and B use this: 260 gym members, nobody has labelled them -----
#      visits per month  x  average minutes per visit
def load_gym(seed=11):
    r = np.random.default_rng(seed)
    a = np.c_[r.normal(19, 3.0, 95), r.normal(38, 7, 95)]     # frequent, short sessions
    b = np.c_[r.normal(6.5, 2.0, 85), r.normal(95, 14, 85)]   # rare, long sessions
    c = np.c_[r.normal(7.0, 2.1, 80), r.normal(35, 7, 80)]    # rare, short sessions
    X = np.vstack([a, b, c])
    truth = np.r_[np.zeros(95), np.ones(85), np.full(80, 2)].astype(int)
    X[:, 0] = X[:, 0].clip(1, None); X[:, 1] = X[:, 1].clip(10, None)
    i = r.permutation(len(X))
    return X[i], truth[i]

GYM, GYM_TRUTH = load_gym()
print("gym members:", len(GYM))
print(f"visits   {GYM[:,0].min():5.1f} to {GYM[:,0].max():5.1f}   spread {GYM[:,0].std():6.2f}")
print(f"minutes  {GYM[:,1].min():5.1f} to {GYM[:,1].max():5.1f}   spread {GYM[:,1].std():6.2f}")

---
# Part A · Running the loop

## Question 1

Write the two steps of the algorithm yourself.

- `assign(Z, centres)` — return the index of the nearest centre for every point
- `move(Z, labels, k)` — return each centre moved to the mean of its own points
- `inertia_of(Z, labels, centres)` — total **squared** distance from each point to its own centre

Scale the gym data first, drop 3 centres at random, then run **one** assign and **one**
move, printing the inertia after each.

In [ ]:
Z = StandardScaler().fit_transform(GYM)

def assign(Z, centres):
    d = ((Z[:, None, :] - centres[None, :, :]) ** 2).sum(axis=2)
    return ...

def move(Z, labels, k):
    return np.array([... for j in range(k)])

def inertia_of(Z, labels, centres):
    return float(...)

rng = np.random.default_rng(0)
centres = Z[rng.choice(len(Z), 3, replace=False)].copy()

labels = assign(Z, centres)
print("after the first assignment:", round(inertia_of(Z, labels, centres), 2))
centres = move(Z, labels, 3)
print("after moving the centres: ", round(inertia_of(Z, labels, centres), 2))

## Question 2

Wrap those two steps in a loop that stops when an assignment changes nobody's mind.
Print the inertia and the number of points that changed cluster at each step.

Then fit `KMeans(3, n_init=10, random_state=0)` on the same scaled data and check you got
the same inertia.

In [ ]:
def my_kmeans(Z, k, seed=0, max_iter=100):
    rng = np.random.default_rng(seed)
    centres = Z[rng.choice(len(Z), k, replace=False)].copy()
    prev = None
    for step in range(max_iter):
        labels = assign(Z, centres)
        ...
    return labels, centres

my_labels, my_centres = my_kmeans(Z, 3)

sk = KMeans(3, n_init=10, random_state=0).fit(Z)
print("mine   ", ...)
print("sklearn", round(sk.inertia_, 2))

## Question 3

Two facts about the algorithm:

- **(i)** Inertia never rises, so the loop is guaranteed to stop.
- **(ii)** The loop is **not** guaranteed to find the lowest possible inertia.

Explain why each is true, in your own words. Then say what practical step (ii) forces you
to take, and name the sklearn argument that takes it for you.

*Your answer:*

---
# Part B · Choosing k

## Question 4

Nobody told you there were three gym groups. Sweep `k` from 1 to 10 on the scaled gym
data and print, for each `k`, the inertia, the **drop** from the previous `k`, and the
silhouette.

Which `k` does the elbow suggest? Which does the silhouette pick?

In [ ]:
rows = []
for k in range(1, 11):
    km = KMeans(k, n_init=10, random_state=0).fit(Z)
    sil = ... if k > 1 else np.nan
    rows.append((k, km.inertia_, sil))

print(f"{'k':>3}{'inertia':>10}{'drop':>9}{'silhouette':>12}")
for i, (k, inert, sil) in enumerate(rows):
    drop = "" if i == 0 else f"{rows[i-1][1] - inert:9.1f}"
    print(f"{k:>3}{inert:>10.1f}{drop}{'' if k == 1 else f'{sil:12.4f}'}")

## Question 5

Now a famous dataset with the labels stripped off. Load `load_iris()`, keep only `X`,
scale it, and sweep `k` from 2 to 6 printing the silhouette.

Which `k` does the silhouette choose?

In [ ]:
Xi, yi = load_iris(return_X_y=True)
Zi = StandardScaler().fit_transform(Xi)

for k in range(2, 7):
    km = KMeans(k, n_init=10, random_state=0).fit(Zi)
    print(f"  k={k}   silhouette {...:.4f}")

## Question 6

The iris dataset has **three** real species — that is what `y` holds.

Compare `adjusted_rand_score(y, labels)` for `k = 2` and `k = 3`, and explain why the
silhouette chose the number it did. Which answer would you report?

In [ ]:
for k in [2, 3]:
    km = KMeans(k, n_init=10, random_state=0).fit(Zi)
    print(f"  k={k}   silhouette {...:.4f}   agreement with the species {...:.4f}")

---
# Part C · Breaking it

## Question 7

Load `load_wine()` — 178 wines, 13 chemical measurements, 3 real cultivars.

Cluster it with `k=3` **twice**: once on the raw columns, once scaled. For each, print the
**silhouette** and the **agreement with the true cultivars**.

In [ ]:
Xw, yw = load_wine(return_X_y=True)
Zw = StandardScaler().fit_transform(Xw)

for name, D in [("raw", Xw), ("scaled", Zw)]:
    km = KMeans(3, n_init=10, random_state=0).fit(D)
    print(f"  {name:7} silhouette {...:.4f}   agreement {...:.4f}")

## Question 8

Look again at the two silhouette scores you just printed. The **raw** clustering scored
**higher** than the scaled one, while being far worse.

Explain how that is possible, and say what it means for using the silhouette to make
decisions.

*Your answer:*

## Question 9

Back to the gym data. Run `KMeans(k, n_init=1, init='random')` for 30 different
`random_state` values and count how many distinct inertias come back, and how many runs
got stuck in a worse one.

Do it **twice: once at `k=3` and once at `k=5`**. The difference between the two is the
point of the question.

In [ ]:
for k in [3, 5]:
    vals = {s: round(KMeans(k, n_init=1, init="random", random_state=s).fit(Z).inertia_, 2)
            for s in range(30)}
    distinct = sorted(set(vals.values()))
    best = min(distinct)
    stuck = ...
    print(f"k={k}:  {len(distinct)} distinct answers   stuck {stuck}/30   "
          f"best {best}   worst {max(distinct)}")

## Question 10

Generate `make_moons(300, noise=0.06, random_state=0)` and cluster it with `k=2`.

Print the silhouette and the agreement with the true crescents, and plot the result beside
the truth. Say in one line why K-Means cannot do this.

In [ ]:
Xm, ym = make_moons(300, noise=0.06, random_state=0)
km = KMeans(2, n_init=10, random_state=0).fit(Xm)

print("silhouette", ...)
print("agreement ", ...)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
show(Xm, ym, title="what is really there", ax=axes[0])
show(Xm, km.labels_, km.cluster_centers_, "what K-Means does", ax=axes[1])
plt.tight_layout(); plt.show()

---
# Part D · Using it

## Question 11

Cluster the gym data with `k=3` on the scaled columns, then build a profile table showing,
for each cluster: how many members, their average **visits per month**, their average
**minutes per visit**, and their average **total minutes per month**.

Then name the three groups.

In [ ]:
km = KMeans(3, n_init=10, random_state=0).fit(Z)

df = pd.DataFrame(GYM, columns=["visits", "minutes"])
df["cluster"] = km.labels_
df["total_minutes"] = ...

profile = df.groupby("cluster").agg(
    members=("visits", "size"),
    visits_per_month=("visits", "mean"),
    minutes_per_visit=("minutes", "mean"),
    total_minutes=("total_minutes", "mean")).round(1)
print(profile)

## Question 12

Look at the `total_minutes` column in your profile.

Write **one sentence** a gym manager could act on this week — something that is true of
the data, specific about which group, and names a concrete action. Then say what you would
check before trusting it.

*Your sentence:*


*What you would check first:*

---

## Submission

Run **Cell → Run All** on a fresh kernel and confirm there are no errors, then upload the
notebook. Questions 3, 8, 11 and 12 are marked mostly on the explanation — a correct number
with no reasoning gets little credit.

Solutions will be released with Day 44.